# Analyse elevation changes of Kverkjökull using ArcticDEM

In [ ]:
import os
import pdemtools as pdt
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np
import ultraplot as uplt

## Load the RGI shapefiles of Iceland's glaciers and find Kverkjökull

In [ ]:
rgi = gpd.read_file("data/RGI/RGI2000-v7.0-G-06_iceland.shp")
glacier = rgi.loc[rgi.rgi_id == "RGI2000-v7.0-G-06-00373"]  # Kverkjökull ID number in RGI v7.0

## Reproject the glacier outlines to the Polar Stereographic projection (EPSG:3413)

In [ ]:
glacier_c = glacier.to_crs("EPSG:3413")
rgi_c = rgi.to_crs("EPSG:3413")

## Define the Area Of Interest (AOI) and non-glacierised regions for co-registration

In [ ]:
# Select the desired padding around the glacier
padding=1.4e3  # 1.4 km is specifically chosen here to not cover Dyngjujökull and Brúarjökull

In [ ]:
bbox = glacier_c.buffer(padding).envelope

In [ ]:
# Clip the RGI region's glacier geometries to the AOI.
rgi_c = gpd.clip(rgi.to_crs("EPSG:3413"), bbox)

# Join all the glacier polygons within the AOI.
rgi_c_union = gpd.GeoDataFrame(index=[0], crs="EPSG:3413", geometry=[rgi_c.convex_hull.union_all()])

# Define the bedrock as the area outside of the glaciers within the AOI.
bedrock = bbox.envelope.difference(rgi_c_union, align=False)

In [ ]:
fig, ax = uplt.subplots()
rgi_c.plot(ax=ax, color="tab:blue", alpha=0.5, edgecolor="k")#, label="Glaciers")
glacier_c.plot(ax=ax, color="tab:blue", edgecolor="k")#, label="Kverkjökull")
bedrock.plot(ax=ax, color="tab:orange", alpha=0.5)#, label="Ice-free area")
bbox.plot(ax=ax, color="none", edgecolor="r")#, label="Search area")
ax.legend(loc="ur")

## Search for ArcticDEM and REMA strips

In [ ]:
gdf = pdt.search(
    dataset='arcticdem',
    bounds = bbox.bounds.values[0],
    years = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    months = [6,7,8,9],
    baseline_max_hours = 24,
    sensors=['WV03', 'WV02', 'WV01'],
    accuracy=2,
    min_aoi_frac = 0.5,
)

if gdf is not None:
    print(f'{len(gdf)} strips found')

## Preview scenes and select the best ones

In [ ]:
def scene_to_preview(gdf, i=0):
    # =================
    # SCENE TO PREVIEW
    # i: scene index
    # =================
    
    preview = pdt.load.preview(gdf.iloc[[i]], bbox.bounds.values[0])
    
    fig, ax = plt.subplots(layout='constrained')
    preview.plot.imshow(cmap='Greys_r', add_colorbar=False)
    
    gdf.iloc[[i]].plot(ax=ax, fc='none', ec='tab:red')
    
    ax.set_title(gdf.iloc[[i]].pdt_datetime1.dt.date.values[0])
    plt.show()

In [ ]:
for i in range(len(gdf)):
    print(i, gdf.pdt_year.iloc[[i]].values[0])
    scene_to_preview(gdf=gdf, i=i)

In [ ]:
# Select the best scenes based on the preview above.
selected_scenes = [0, 1, 5, 7, 9, 11, 12, 16]  # 15 and 17 also usable, but those are the days before and after 16.
gdf_sel = gdf.iloc[selected_scenes]

In [ ]:
gdf_sel.pdt_datetime_mean.dt.date

## Download the DEMs

In [ ]:
def download_arcticdem(gdf_row):
    dir_dem = os.path.join("data", "DEM")
    if not os.path.exists(dir_dem):
        os.mkdir(dir_dem)

    out_fpath = os.path.join(dir_dem, f'{gdf_row.pdt_id.values[0]}.tif')
    if not os.path.exists(out_fpath):
        # dem = pdt.load.from_search(gdf_row, bounds=glacier_c.bounds.values[0], bitmask=True, pad=True).compute()
        dem = pdt.load.from_search(gdf_row, bounds=bbox.bounds.values[0], bitmask=True, pad=True).compute()
        dem.rio.to_raster(out_fpath, compress='ZSTD', predictor=3, zlevel=1)
    else:
        # dem = pdt.load.from_fpath(out_fpath, bounds=glacier_c.bounds.values[0], pad=True)
        dem = pdt.load.from_fpath(out_fpath, bounds=bbox.bounds.values[0], pad=True)
    return dem

In [ ]:
for i in selected_scenes:
    dem = download_arcticdem(gdf_sel.loc[[i]])

## Coregister the DEMs and compute differences to view

In [ ]:
vrange = 10  # Vertical difference limits for plotting in metres
dem_reg_ind = 16  # Select which DEM to use as the ground truth for coregistration.

In [ ]:
def dz_coreg(gdf_sel=gdf_sel, i=0, j=1, dem_reg_ind=dem_reg_ind, return_stats=True):
    dem_ref = download_arcticdem(gdf_sel.loc[[dem_reg_ind]])
    dem1 = download_arcticdem(gdf_sel.iloc[[i]])
    dem2 = download_arcticdem(gdf_sel.iloc[[j]])

    if i == dem_reg_ind:
        dem1_coreg_bedrock = dem_ref.copy()
    else:
        # print(gdf_sel.iloc[[i]])
        dem1_coreg_bedrock, metadata_coreg_bedrock = dem1.pdt.coregister_dems(
            dem_ref,
            stable_mask=pdt.data.bedrock_mask_from_vector(
                bedrock, 
                dem_ref
            ),
            return_stats=return_stats,
        )

    if j == dem_reg_ind:
        dem2_coreg_bedrock = dem_ref.copy()
    else:
        # print(gdf_sel.iloc[[j]])
        dem2_coreg_bedrock, metadata_coreg_bedrock = dem2.pdt.coregister_dems(
            dem_ref,
            stable_mask=pdt.data.bedrock_mask_from_vector(
                bedrock, 
                dem_ref
            ),
            return_stats=return_stats,
        )

    return dem2_coreg_bedrock - dem1_coreg_bedrock

In [ ]:
def df2str(df, col, strformat="%Y/%m/%d"):
    return df[col].dt.strftime(strformat).values[0]

In [ ]:
def plot_dem_diff(ax, gdf_sel=gdf_sel, i=0, j=1, dem_reg_ind=9):
    dz = dz_coreg(gdf_sel=gdf_sel, i=i, j=j, dem_reg_ind=dem_reg_ind)

    dz_plot = dz.plot(
        ax=ax, cmap="coolwarm_r", vmin=-vrange, vmax=vrange, levels=255, 
        add_colorbar=False
        # cbar_kwargs={"label": "Elevation change [m]", "ticks": [-10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10], "minorticks": []}
    )
    dz_plot.set_label(None)
    rgi_c.plot(ax=ax, color="none", edgecolor="k", alpha=0.5)
    glacier_c.plot(ax=ax, color="none", edgecolor="k")

    return dz_plot

### Subplots of all selected DEM differences

In [ ]:
uplt.rc.labelsize="xx-large"
uplt.rc.update()

rows2drop = [11]  # index of rows that should not be used to compute DEM differences.
inds = np.arange(len(gdf_sel))
while len(rows2drop) > 0:
    inds = inds[gdf_sel.index != rows2drop[0]]
    rows2drop.remove(rows2drop[0])

fig3, axs3 = uplt.subplots(nrows=1, ncols=len(inds)-1, span=True, share=True)

# Iterate through the selected DEMs.
for k, i in enumerate(inds[:-1]):
    # if i != 6:
    #     continue
    title = f"{df2str(gdf_sel.iloc[[i]], "pdt_datetime1")} - {df2str(gdf_sel.iloc[[i+1]], "pdt_datetime1")}"
    print("  ---  " + title + "  ---  ")
    m = plot_dem_diff(ax=axs3[k], i=i, j=i+1, dem_reg_ind=dem_reg_ind)
    axs3[k].format(title=title)

fig3.colorbar(m, label="Elevation change [m]", ticks=5, minorticks=1, loc="r", length=0.8, vmin=-vrange, vmax=vrange, extend="both",
             labelsize="xx-large", ticklabelsize="x-large")
axs3.format(
    abc=True, abcsize="xx-large", abcloc="ur",
    xlabel=r"Easting [$\times 10^6$ m]", ylabel=r"Northing [$\times 10^6$ m]", 
    xformatter=lambda x, pos:"{:.3f}".format(x/1e6), yformatter=lambda x, pos:"{:.3f}".format(x/1e6),
    titlesize="xx-large", labelsize="xx-large", ticklabelsize="x-large"
)
uplt.show()

In [ ]:
fig3.savefig("fig_03.png")